# Chapter 3 (leptonic) — Notebook 2: Lepton–b-jet pairing

**Goals**

- Pick which of the two b-tagged jets belongs to the **leptonic** top decay using smallest ΔR(ℓ, b).

In [ ]:
%matplotlib inline
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from topmass import io, kinematics, selection, plotting, fitting, neutrino, pairing, weights, style
from topmass.constants import M_W, M_TOP

In [ ]:
io.setup()                                  # select release 2025e-13tev-beta
samples = io.build_samples()                # skim '3J1LMET30', https
events = io.load_process('ttbar', samples, fraction=0.1)
print('Number of events:', len(events))

In [ ]:
cuts = selection.SemilepCuts(n_jets_min=4, n_bjets_min=2)
events = events[selection.semilep_preselection(events, cuts)]

lep = kinematics.leading_lepton(events)
jets = kinematics.jet_vectors(events)
is_b = events.jet_btag_quantile >= cuts.btag_quantile_min
b_jets = jets[is_b]
keep = ak.num(b_jets) >= 2
b_jets, lep = b_jets[keep][:, :2], lep[keep]

b_lep, b_had = pairing.assign_bjets(b_jets, lep)
print('ΔR(ℓ, b_lep) mean:', float(ak.mean(b_lep.deltaR(lep))))
print('ΔR(ℓ, b_had) mean:', float(ak.mean(b_had.deltaR(lep))))

## ✏️ Your turn 2.1

▶️ Change the binning and re-run.

This overlays ΔR(ℓ, b_lep) and ΔR(ℓ, b_had) on the same axes. The leptonic-side b-jet is chosen to
be the one closer to the lepton, so its distribution should peak at smaller ΔR — does the plot
confirm the heuristic?

> **Stretch (optional):** change `DR_BINS` to see how clean the separation looks at finer/coarser binning.

In [ ]:
DR_BINS = 40    # ✏️ try 20, 40, 60

plt.hist(ak.to_numpy(b_lep.deltaR(lep)), bins=DR_BINS, range=(0, 5), histtype='step', label='ΔR(ℓ, b_lep)')
plt.hist(ak.to_numpy(b_had.deltaR(lep)), bins=DR_BINS, range=(0, 5), histtype='step', label='ΔR(ℓ, b_had)')
plt.xlabel('ΔR(lepton, b-jet)'); plt.ylabel('Events'); plt.legend()